# Installation
install torch for you GPU/MPS/CPU https://pytorch.org/get-started/locally/



In [10]:
%load_ext autoreload
%autoreload 2
try:
    import deepali
    import TPTBox
except Exception:
    %pip install TPTBox ruamel.yaml configargparse
    %pip install hf-deepali
    %pip install nnunetv2

    import deepali
    import TPTBox
from pathlib import Path
from typing import Literal

import pandas as pd
import torch
from TPTBox import NII, POI_Global, to_nii
from TPTBox.core.vert_constants import Full_Body_Instance

from treg.angle import compute_angles
from treg.basics import resolve_device


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
dataset = "/media/data/robert/code/TReg/private/dataset-treg"
folder = "rawdata"
dicom_folder = None

dataset_id = 11
ddevice: Literal["cpu", "cuda", "mps"] = "cuda"
gpu_id = 0  # Only used for cuda

run_only_one = True
subject_name = None




In [12]:
# Atlas
atlas_seg_file: str | Path = "data/sub-atlas_seg-VIBESeg-12_msk.nii.gz"  # default is a left leg
atlas_file: str | Path = "data/sub-atlas_seg-poi_poi.json"
atlas_seg_subdivided_file: str | Path | None = "data/sub-atlas_seg-subregion_msk.nii.gz"


In [13]:
from TPTBox.core.dicom.dicom_extract import extract_dicom_folder

# TODO Step 0 export to a nii dataset
if dicom_folder is not None:
# Uncomment and change paths
    extract_dicom_folder(
        dicom_folder=Path("/media/data/robert/code/TReg/private/Pat0001"),
        dataset_path_out=dicom_folder,
        use_session=True)

In [14]:
import numpy as np
from TPTBox import BIDS_FILE, BIDS_Global_info
from TPTBox.segmentation import run_vibeseg
from TPTBox.stitching import stitching

# -----------------------------------------------------------------------------
# Loop over the dataset an merge images with the same kernal. Segment all images.
# The dataset must be in the BIDS-Format and the dicom header must be provided as a json with the same file name and path. extract_dicom_folder creates such a dataset
# Will select best complete Leg from a 3 view stich (pelvic, knee, foot). Ignores type of recon kernal.
# -----------------------------------------------------------------------------
bgi = BIDS_Global_info(dataset, parents=folder)

# Dictionary storing the final selected image/segmentation per subject
subs = {}

# -----------------------------------------------------------------------------
# Iterate through all subjects in the BIDS dataset
#
# sub:
#     subject ID string
#
# subj:
#     subject-specific BIDS object
# -----------------------------------------------------------------------------
for sub, subj in bgi.iter_subjects(sort=True):

    # Create a query object to search files belonging to this subject
    q = subj.new_query(flatten=True)

    # Keep only CT images
    q.filter_format("ct")

    # Keep only NIfTI files
    q.filter_filetype("nii.gz")

    # Optional:
    # Restrict to isotropic acquisitions only
    # q.filter("acq","iso",required=False)

    # Ignore localizer scans
    # (localizer are low-quality planning scans)
    q.filter("part", lambda x: x != "localizer", required=False)

    # -------------------------------------------------------------------------
    # Dictionaries grouping images by acquisition/kernel properties
    #
    # img:
    #     stores original image files
    #
    # segs:
    #     stores segmentation outputs
    # -------------------------------------------------------------------------
    img: dict[str, list[BIDS_FILE]] = {}
    segs: dict[str, list] = {}

    # -------------------------------------------------------------------------
    # Loop over all matching CT files
    # -------------------------------------------------------------------------
    for file in q.loop_list(sort=True):
        # Create output segmentation path
        # Example:
        # sub-001_seg-VIBESeg-XYZ_msk.nii.gz
        out_file = file.get_changed_path("nii.gz","msk",info={"seg": f"VIBESeg-{dataset_id}"})
        # ---------------------------------------------------------------------
        # Run automatic segmentation
        #
        # gpu=0:
        #     use GPU index 0
        #
        # ddevice="cuda":
        #     inference on CUDA GPU
        #
        # dataset_id:
        #     segmentation model identifier
        # ---------------------------------------------------------------------
        out = run_vibeseg(file,out_file,gpu=gpu_id,ddevice=ddevice,dataset_id=dataset_id)
        # Load accompanying JSON metadata
        j: dict = file.open_json()
        # ---------------------------------------------------------------------
        # Build a grouping key ("kernel")
        #
        # Images with identical:
        # - convolution kernel
        # - session
        # - acquisition
        # - part
        #
        # are stitched together later.
        # ---------------------------------------------------------------------
        kernel = j.get("ConvolutionKernel", "ct")
        kernel += "-" + str(file.get("ses", ""))
        kernel += "-" + str(file.get("acq", ""))
        kernel += "-" + str(file.get("part", ""))

        # Initialize lists if key does not exist yet
        if kernel not in img:
            img[kernel] = []
            segs[kernel] = []
        # Store image and segmentation
        img[kernel].append(file)
        segs[kernel].append(out)

    # -------------------------------------------------------------------------
    # Stitch images belonging to the same acquisition group
    # -------------------------------------------------------------------------
    for name, l_images in img.items():

        # Nothing to stitch if only one image exists
        if len(l_images) <= 1:
            continue

        # Build sequence identifier from all sequence names
        seq = "-".join(sorted([str(a.get("sequ")) for a in l_images]))

        # Output stitched CT image
        out = l_images[0].get_changed_path("nii.gz","msk","rawdata",info={"sequ": f"stiched-{seq}"})
        # Output stitched segmentation
        out_seg = l_images[0].get_changed_path("nii.gz","msk",info={"sequ": f"stiched-{seq}","seg": f"VIBESeg-{dataset_id}"})
        # Output blending/ramp mask
        #
        # The ramp image stores blending weights used during stitching.
        out_ramp = l_images[0].get_changed_path("nii.gz","ramp",info={"sequ": f"stiched-{seq}","seg": f"VIBESeg-{dataset_id}"},non_strict_mode=True)
        # ---------------------------------------------------------------------
        # Stitch original CT volumes
        # ---------------------------------------------------------------------
        if not out.exists():
            stitching(l_images,out,is_ct=True,verbose=True,verbose_stitching=True,dtype=np.int16,store_ramp=True,ramp_path=out_ramp)
        # ---------------------------------------------------------------------
        # Stitch segmentation masks
        # ---------------------------------------------------------------------
        if not out_seg.exists():
            stitching(segs[name],out_seg,is_seg=True,verbose=True,verbose_stitching=True)
            # Compress datatype to smallest possible unsigned integer
            # to reduce file size
            to_nii(out_seg, True).set_dtype("smallest_uint").save(out_seg)
        # ---------------------------------------------------------------------
        # Select the "best" acquisition per subject
        #
        # Priority:
        #   1. isotropic ("iso")
        #   2. axial ("ax")
        #   3. everything else
        #
        # Additionally:
        #   only keep cases where labels 11-14 exist
        # ---------------------------------------------------------------------
        if (sub not in subs or (l_images[0].get("acq", "") == "iso" and subs[sub]["acq"] != "iso") or (l_images[0].get("acq", "") == "ax" and subs[sub]["acq"] not in ["iso", "ax"])):
            # Get all labels present in segmentation
            u = to_nii(out_seg, True).unique()
            # Require labels 11,12,13,14 to exist
            # version 12 may additionally require label 100
            if all(a in u for a in range(11, 15)) or all(a in u for a in range(111, 115)):
                # Store final selected subject entry
                subs[sub] = {
                    "img": out,                       # stitched CT image
                    "seg": out_seg,                   # stitched segmentation
                    "dataset": l_images[0].dataset,   # originating dataset
                    "bin_msk": out_ramp,              # stitching ramp mask
                    "acq": l_images[0].get("acq", "") # acquisition type
                }

[!] Unknown file_type pdf in file sub-1339434_sequ-9999_report.pdf


In [15]:

subject_names = ([next(iter(subs.keys()))] if subject_name is None else [subject_name]) if run_only_one else list(subs.keys())
print(subject_names)

['1339434']


In [ ]:
#############################################
################# Input #####################
#############################################
all_tasks = []
for subject_name in subject_names:
    files = subs[subject_name]
    #{"img":out,"seg":out_seg,"dataset":l[0].dataset}
    ds = BIDS_FILE(files["img"],files["dataset"])
    sides = ["left","right"]
    for side in sides:
        files[side] = {}
        files[side]["target_out_poi"] = ds.get_changed_path("json","poi",info={"desc":"atlas","seg":side})
        files[side]["target_out_subdivided"] = ds.get_changed_path("nii.gz","msk",info={"desc":"atlas","seg":side},additional_folder=side)
        files[side]["target_out_angle"] = ds.get_changed_path("nii.gz","msk",info={"desc":"angle","seg":side},additional_folder=side)
        files[side]["target_out_angle2"] = ds.get_changed_path("nii.gz","msk",info={"desc":"angle-veerman","seg":side})
        files[side]["mirror"] ="right" in side

## Generate Segmentation


In [ ]:
from treg.basics import run_all

run_all(files, sides, ddevice=ddevice, gpu=gpu_id)

[TREG] load mask
[TREG] sub-1339434_sequ-stiched-2-4-6_acq-iso_seg-VIBESeg-11_msk.nii.gz side='left'
[10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 36, 37, 38, 49, 51, 52, 53, 54, 55, 56, 57, 58, 59, 113, 114, 115, 116, 117, 118, 119, 120, 121] [113, 114, 115, 116]
unique [113, 114, 115, 116] [113, 114, 115, 116]


100%|██████████| 4/4 [00:00<00:00,  8.90it/s]


crop
[ ] Padd ((38, 38), (38, 38), (38, 38)); mode='constant', {'constant_values': 0}
iter 0: using crop (50)
point registration ok
[(113, 40), (114, 40), (115, 40), (116, 40)]
[*] ('L', 'A', 'S')
[*] used POI: [(113, 40), (114, 40), (115, 40), (116, 40)]
[*] key    |fixed points           |moved points after     |moved points before    |delta fixed/moved      |distF|distM|
[*] (113, 40)|  102.1, -138.2,  585.6|  102.4, -141.3,  596.7|   91.8,  166.8, -933.0|   -0.3,    3.1,  -11.1|     |     |
[*] (114, 40)|   80.8, -180.9,  425.5|   81.7, -175.4,  430.4|   79.5,  138.5,-1101.2|   -0.9,   -5.5,   -4.9|167.1|171.0|
[*] (115, 40)|   83.3, -137.2,  266.8|   84.6, -139.4,  241.5|   68.0,  185.4,-1287.3|   -1.3,    2.2,   25.3|164.6|192.3|
[*] (116, 40)|  112.0, -122.3,  179.5|  109.5, -122.6,  189.0|   85.6,  212.2,-1338.6|    2.5,    0.3,   -9.5|93.1 |60.5 |
[*] Error avg registration error-vector length:  13.552
[*] Error avg point-distances:  21.400
Target image pyramid:
- Level 3: siz

   264: 0.004474 (loss), 0.00416[1*seg], 0.00000[0.001*Tether], 0.00028[0.01*Dice], 0.00003[1e-05*be]:  18%|█▊        | 265/1500 [00:06<00:29, 41.27it/s]


Subdivided control point grid in 0.001s


...

100%|██████████| 4/4 [00:00<00:00,  8.92it/s]


crop
[ ] Padd ((38, 38), (38, 38), (38, 38)); mode='constant', {'constant_values': 0}
iter 0: using crop (50)
point registration ok
[(113, 40), (114, 40), (115, 40), (116, 40)]
[*] ('L', 'A', 'S')
[*] used POI: [(113, 40), (114, 40), (115, 40), (116, 40)]
[*] key    |fixed points           |moved points after     |moved points before    |delta fixed/moved      |distF|distM|
[*] (113, 40)|   81.9, -140.1,  569.3|   82.1, -143.4,  592.6|   91.8,  166.8, -933.0|   -0.2,    3.3,  -23.3|     |     |
[*] (114, 40)|   61.0, -189.6,  424.6|   62.1, -183.2,  427.5|   79.5,  138.5,-1101.2|   -1.1,   -6.4,   -2.9|154.4|171.0|
[*] (115, 40)|   59.1, -150.7,  264.5|   62.0, -152.5,  237.7|   68.0,  185.4,-1287.3|   -2.9,    1.8,   26.8|164.8|192.3|
[*] (116, 40)|   89.7, -134.5,  183.7|   85.6, -136.0,  184.5|   85.6,  212.2,-1338.6|    4.1,    1.5,   -0.8|87.9 |60.5 |
[*] Error avg registration error-vector length:  15.525
[*] Error avg point-distances:  23.833
Target image pyramid:
- Level 3: siz

   391: 0.005504 (loss), 0.00504[1*seg], 0.00000[0.001*Tether], 0.00040[0.01*Dice], 0.00006[1e-05*be]:  26%|██▌       | 392/1500 [00:09<00:25, 43.40it/s]


Subdivided control point grid in 0.002s


    13: 0.006994 (loss), 0.00648[1*seg], 0.00000[0.001*Tether], 0.00038[0.01*Dice], 0.00013[1e-05*be]:   1%|          | 14/1500 [00:00<00:35, 42.25it/s]